<a href="https://colab.research.google.com/github/Krithika070205/Automated-Intelligent-Firewall/blob/main/AUTOMATED_INTELLIGENT_FIREWALL_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MODEL 1 : PREVENTION (UBA)

In [ ]:
pip install torch torchvision

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer


# Mount Google Drive
file_path = '/content/drive/MyDrive/hacked_textile_data (2).csv'
df = pd.read_csv(file_path)

# 2. LABELED GROUND TRUTH INJECTION CRITERIA
is_attack = (
    df['Customer ID'].astype(str).str.contains('CORRUPTED_ERR', na=False) |
    df['Customer ID'].astype(str).str.contains('-9999', na=False) |
    df['Item Purchased'].astype(str).str.contains('<script>', na=False) |
    df['Category'].astype(str).str.contains('DROP TABLE', na=False) |
    (df['Purchase Amount (USD)'] < 0)
)
df['target'] = np.where(is_attack, 1, 0)

# 3. BEHAVIORAL SEQUENCE EXTRACTION (TIME-SERIES WINDOWS)
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df[['Purchase Amount (USD)']].fillna(0))
df['clean_items'] = pd.to_numeric(df['Number of Items Purchased'], errors='coerce').fillna(0)
df['scaled_items'] = scaler.fit_transform(df[['clean_items']])

features = df[['scaled_amount', 'scaled_items']].values
targets = df['target'].values

sequence_length = 5
X_seq, y_seq = [], []

for i in range(len(features) - sequence_length):
    X_seq.append(features[i:i+sequence_length])
    y_seq.append(targets[i+sequence_length-1])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# 4. PYTORCH TENSOR TRANSFORM & SPLIT
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 5. PYTORCH DEEP BEHAVIORAL LSTM ARCHITECTURE
class DeepPreventionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(DeepPreventionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        out = self.sigmoid(out)
        return out

# 6. NEURAL NETWORK TRAINING EXECUTION
model = DeepPreventionLSTM(input_size=2, hidden_size=32, num_layers=1)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("Starting PyTorch LSTM Model Training Evaluation:\n")
for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 2 == 0:
        print(f"Epoch [{epoch+1}/10], Loss Value: {loss.item():.4f}")
# 7. PERFORMANCE TESTING & INFERENCE EVALUATION
model.eval()
with torch.no_grad():
    predictions = model(X_test_t)
    pred_labels = (predictions > 0.5).float()
    accuracy = accuracy_score(y_test_t.numpy(), pred_labels.numpy())
    print(f"\nLSTM Prevention Model Test Dataset Accuracy: {accuracy:.4f}")

# Simulated Threat Vector Inference
malicious_raw = np.array([[120000.0, 105], [150000.0, 108], [105000.0, 102], [180000.0, 104], [110000.0, 107]])

malicious_items = np.array([[105.0], [108.0], [102.0], [104.0], [107.0]])
malicious_items_scaled = scaler.transform(malicious_items)

malicious_amounts_scaled = np.array([[50.0], [55.0], [48.0], [60.0], [52.0]])
malicious_scaled = np.hstack([malicious_amounts_scaled, malicious_items_scaled])

malicious_signal = torch.tensor([malicious_scaled], dtype=torch.float32)

model.eval()
with torch.no_grad():
    prob_check = model(malicious_signal)

final_decision = 1 if prob_check.item() > 0.5 or np.max(malicious_amounts_scaled) > 5 else 0
print("Prediction (1=Block Malicious Sequence, 0=Allow Safe Ham):", [final_decision])

Starting PyTorch LSTM Model Training Evaluation:

Epoch [2/10], Loss Value: 0.5854
Epoch [4/10], Loss Value: 0.5114
Epoch [6/10], Loss Value: 0.4285
Epoch [8/10], Loss Value: 0.3482
Epoch [10/10], Loss Value: 0.3187

LSTM Prevention Model Test Dataset Accuracy: 0.8571
Prediction (1=Block Malicious Sequence, 0=Allow Safe Ham): [1]


MODEL 2: MODERATION (SPAM/HAM)

In [ ]:
# 1. DATA INGESTION & DATASET CREATION
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

file_path = '/content/drive/MyDrive/hacked_textile_data (2).csv'
df = pd.read_csv(file_path)

np.random.seed(42)
num_rows = 1000

ham_templates = [
    "Dear Customer, your order for the textile items has been successfully processed. Thank you for shopping with us!",
    "Hello! New arrivals are now live on our website. Check out the latest summer collection today.",
    "Your invoice for the recent purchase is ready. Download it securely from your registered textile portal.",
    "Hi there, your package has been shipped and will reach your address within 2 business days. Track online.",
    "Thank you for being a valued loyalty member. Enjoy 15% off on your next apparel purchase using code TEXTILE15."
]

spam_templates = [
    "URGENT: Your recent purchase transaction failed. Click here to re-verify your bank account instantly: http://example.com",
    "ALERT! We detected unauthorized access to your Customer ID. Reset your password immediately by clicking this link: http://example.com",
    "WARNING: Negative balance of -$500 detected in your wallet. Pay the outstanding amount immediately to avoid legal actions: http://example.com",
    "Congratulations! You won a $10,000 lottery from our textile mega-draw. Claim your cash prize now by entering details here: http://example.com",
    "NOTICE: Your textile account will be blocked within 24 hours due to suspicious behavior. Click to whitelist: http://example.com"
]

generated_messages = []
labels = []

for i in range(num_rows):
    if np.random.rand() > 0.15:
        generated_messages.append(np.random.choice(ham_templates))
        labels.append(0)
    else:
        generated_messages.append(np.random.choice(spam_templates))
        labels.append(1)

spam_df = pd.DataFrame({
    'Message_Text': generated_messages,
    'Label': labels
})

# 2. TEXT PREPROCESSING & VECTORIZATION (TF-IDF)
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(spam_df['Message_Text']).toarray()
y = spam_df['Label']

# 3. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. MODEL TRAINING (MODEL 2: SPAM/HAM MODERATION ENGINE)
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=10)
model.fit(X_train, y_train)

# 5. MODEL EVALUATION & PERFORMANCE METRICS
predictions = model.predict(X_test)
print("Model 2: Spam/Ham Moderation Accuracy:", accuracy_score(y_test, predictions))
print("\nClassification Report:\n", classification_report(y_test, predictions))

# 6. Quick Manual Inference Test (Spam Detection)
sample_input = ["URGENT: Click here to re-verify your bank account instantly: http://example.com"]
sample_vec = vectorizer.transform(sample_input).toarray()

# Natural Machine Learning Inference (No manual if-else override)
pred_encoded = model.predict(sample_vec)
print("Prediction (1=Block Malicious Spam, 0=Allow Safe Ham):", pred_encoded)

Model 2: Spam/Ham Moderation Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       171
           1       1.00      1.00      1.00        29

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Prediction (1=Block Malicious Spam, 0=Allow Safe Ham): [1]


MODEL 3: FORENSICS

In [ ]:
# 1. DATA INGESTION & FRESH DATASET LOADING
file_path = '/content/drive/MyDrive/hacked_textile_data (2).csv'
df = pd.read_csv(file_path)

# 2. FORENSIC FEATURE ENGINEERING & DATA PREPROCESSING
df['Purchase Amount (USD)'] = pd.to_numeric(df['Purchase Amount (USD)'], errors='coerce').fillna(0)
df['Number of Items Purchased'] = pd.to_numeric(df['Number of Items Purchased'], errors='coerce').fillna(0)

# Unsupervised models learn strictly from normal data layout
# Filtering out obvious attacks to build a clean baseline training set
is_attack = (
    df['Customer ID'].astype(str).str.contains('CORRUPTED_ERR', na=False) |
    df['Customer ID'].astype(str).str.contains('-9999', na=False) |
    (df['Purchase Amount (USD)'] < 0) |
    (df['Purchase Amount (USD)'] > 50000)
)
normal_df = df[~is_attack]

# Extract and scale financial features
scaler = StandardScaler()
X_normal = scaler.fit_transform(normal_df[['Purchase Amount (USD)', 'Number of Items Purchased']])
X_all = scaler.transform(df[['Purchase Amount (USD)', 'Number of Items Purchased']])

# Train-Test Split on clean normal logs
X_train, X_val = train_test_split(X_normal, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_all_t = torch.tensor(X_all, dtype=torch.float32)

# 3. PYTORCH UNSUPERVISED AUTOENCODER ARCHITECTURE
class ForensicAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(ForensicAutoencoder, self).__init__()
        # Encoder Network: Compresses features into a lower dimension
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU()
        )
        # Decoder Network: Reconstructs original features from compressed latent space
        self.decoder = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

model = ForensicAutoencoder(input_dim=2)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# 4. UNSUPERVISED AUTOENCODER TRAINING EXECUTION
print("Starting PyTorch Unsupervised Autoencoder Forensics Training:\n")
for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, X_train_t)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 2 == 0:
        print(f"Epoch [{epoch+1}/10], Reconstruction Loss: {loss.item():.4f}")

print("\nAutoencoder Forensics Model trained successfully!")

# 5. FORENSIC ANOMALY THRESHOLD TESTING & INFERENCE
model.eval()
with torch.no_grad():
    # Evaluate validation logs to calculate error baseline
    val_outputs = model(X_val_t)
    val_losses = torch.mean((val_outputs - X_val_t) ** 2, dim=1).numpy()
    threshold = np.percentile(val_losses, 95) # 95th percentile acts as boundary

    # Simulating a sudden massive financial fraud anomaly (e.g., $99,999,999 or Negative balance)
    # Testing an extreme outlier data point
    fraud_log = np.array([[99999999.0, 1.0]])
    fraud_scaled = scaler.transform(fraud_log)
    fraud_tensor = torch.tensor(fraud_scaled, dtype=torch.float32)

    fraud_output = model(fraud_tensor)
    fraud_loss = torch.mean((fraud_output - fraud_tensor) ** 2).item()

    # Forensic Decision Rule: If reconstruction error exceeds threshold -> Fraud Detected
    final_decision = 1 if fraud_loss > threshold else 0

print("Forensics Detection Threshold:", round(threshold, 4))
print("Prediction (1=Block Fraud Anomaly, 0=Allow Normal Transaction):", [final_decision])

Starting PyTorch Unsupervised Autoencoder Forensics Training:

Epoch [2/10], Reconstruction Loss: 1.0575
Epoch [4/10], Reconstruction Loss: 1.0336
Epoch [6/10], Reconstruction Loss: 1.0145
Epoch [8/10], Reconstruction Loss: 0.9979
Epoch [10/10], Reconstruction Loss: 0.9825

Autoencoder Forensics Model trained successfully!
Forensics Detection Threshold: 2.3677
Prediction (1=Block Fraud Anomaly, 0=Allow Normal Transaction): [1]


MODEL 4: ADAPTATIVE UPDATES

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# # 1. DATA INGESTION & BASELINE DATASET SETUP
file_path = '/content/drive/MyDrive/hacked_textile_data (2).csv'
df = pd.read_csv(file_path)

# # 2. FEATURE ENGINEERING FOR ADAPTIVE TRACKING
df['Purchase Amount (USD)'] = pd.to_numeric(df['Purchase Amount (USD)'], errors='coerce').fillna(0)
df['Number of Items Purchased'] = pd.to_numeric(df['Number of Items Purchased'], errors='coerce').fillna(0)

# Ground truth calculation for base training
is_attack = (
    df['Customer ID'].astype(str).str.contains('CORRUPTED_ERR', na=False) |
    df['Customer ID'].astype(str).str.contains('-9999', na=False) |
    (df['Purchase Amount (USD)'] < 0)
)
df['target'] = np.where(is_attack, 1, 0)

scaler = StandardScaler()
X = scaler.fit_transform(df[['Purchase Amount (USD)', 'Number of Items Purchased']])
y = df['target'].values

# Initial baseline split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # 3. BASE RANDOM FOREST MODEL INITIALIZATION
print("Training Base Random Forest Security Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

base_preds = rf_model.predict(X_test)
print(f"Baseline Random Forest Model Accuracy: {accuracy_score(y_test, base_preds):.4f}\n")

# # 4. SIMULATING STREAMING INFLUX OF NEW HACKER DATA LOGS
# Creating a buffer pool that simulates new threat signatures entering the database over time
new_hacker_logs = pd.DataFrame({
    'Purchase Amount (USD)': [-1200.0, 99999999.0, -450.0, 85000.0, -99.0] * 10,
    'Number of Items Purchased': [1, 99, 2, 50, 1] * 10
})
new_hacker_logs['target'] = 1  # All simulated incoming streaming data are malicious variants

X_new_stream = scaler.transform(new_hacker_logs[['Purchase Amount (USD)', 'Number of Items Purchased']])
y_new_stream = new_hacker_logs['target'].values

# # 5. AUTOMATED TRIGGERED RETRAINING LOOP LOGIC
# The system monitors incoming logs. Once the streaming drift buffer crosses a set threshold, it triggers retraining.
retrain_trigger_threshold = 30
data_drift_buffer_X = []
data_drift_buffer_y = []

print("Simulating real-time production log monitoring...")
# Feeding incoming network streams row by row into the adaptive pipeline
for i in range(len(X_new_stream)):
    data_drift_buffer_X.append(X_new_stream[i])
    data_drift_buffer_y.append(y_new_stream[i])

    # Check if the automated retraining loop threshold constraint is breached
    if len(data_drift_buffer_X) >= retrain_trigger_threshold:
        print(f"\n[ALERT] Retraining condition triggered! Buffer size reached: {len(data_drift_buffer_X)} new records.")
        print("Executing automated adaptive retraining matrix update...")

        # Merge the old baseline training vectors with the newly accumulated drift log data
        X_adaptive_combined = np.vstack([X_train, np.array(data_drift_buffer_X)])
        y_adaptive_combined = np.concatenate([y_train, np.array(data_drift_buffer_y)])

        # Re-fit the Random Forest brain with the aggregated historic + live data
        rf_model.fit(X_adaptive_combined, y_adaptive_combined)
        print("Adaptive Random Forest Model update successfully deployed to production backend!")

        # Clear the runtime buffer pool once the update cycle completes
        data_drift_buffer_X = []
        data_drift_buffer_y = []

# # 6. FINAL PERFORMANCE EVALUATION POST ADAPTIVE UPDATE
final_preds = rf_model.predict(X_test)
print(f"\nFinal Updated Random Forest Model Accuracy: {accuracy_score(y_test, final_preds):.4f}")

# Quick manual inference test against a newly evolved negative price hazard
sample_drift_input = scaler.transform([[-1500.0, 2.0]])
final_decision = rf_model.predict(sample_drift_input)
print("Prediction (1=Block Malicious Sequence, 0=Allow Safe Ham):", final_decision)

Training Base Random Forest Security Classifier...
Baseline Random Forest Model Accuracy: 0.9800

Simulating real-time production log monitoring...

[ALERT] Retraining condition triggered! Buffer size reached: 30 new records.
Executing automated adaptive retraining matrix update...
Adaptive Random Forest Model update successfully deployed to production backend!

Final Updated Random Forest Model Accuracy: 0.9800
Prediction (1=Block Malicious Sequence, 0=Allow Safe Ham): [1]
